# Fine-tune Coqui TTS XTTS v2 on Your Voice

This notebook fine-tunes the XTTS v2 model on your voice dataset.

## Before you start:
1. Record 6+ minutes of clean speech using `scripts/record_voice.py`
2. Upload the `voice_dataset/` folder as a Kaggle Dataset
3. Add your dataset to this notebook (right panel -> Input -> Add Dataset)

## Requirements:
- GPU T4 x2 (recommended) or P100
- 6+ minutes of clean audio (more is better)
- Your audio in 22050Hz mono WAV format

In [ ]:
import os
import sys
import subprocess

print("[1/7] Installing PyTorch with CUDA...")
!pip install --upgrade pip -q
!pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu118 -q

In [ ]:
print("[2/7] Installing Coqui TTS and dependencies...")
!pip install TTS soundfile librosa pydub einops transformers tokenizers sentencepiece coqpit -q

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
print("[3/7] Verifying dataset structure...")

import glob
import csv

DEST = "/kaggle/working/voice_dataset"
SOURCE_CANDIDATES = [
    "/kaggle/input/voice-dataset/voice_dataset",
    "/kaggle/input/voice_dataset",
    "/kaggle/input/voice-dataset",
    "/kaggle/working/voice_dataset",
]

source = None
for path in SOURCE_CANDIDATES:
    if os.path.isdir(path) and os.path.isdir(os.path.join(path, "wavs")):
        source = path
        break

if source is None:
    print("ERROR: No dataset found. Check your Kaggle input.")
else:
    if source != DEST:
        import shutil
        if os.path.exists(DEST):
            shutil.rmtree(DEST)
        shutil.copytree(source, DEST)
    
    wav_files = glob.glob(os.path.join(DEST, "wavs", "*.wav"))
    print(f"WAV files: {len(wav_files)}")
    
    total_dur = 0
    import soundfile as sf
    for wf in wav_files[:5]:  # sample first 5 for speed
        total_dur += sf.info(wf).duration
    avg = total_dur / min(5, len(wav_files))
    est_total = avg * len(wav_files) / 60
    print(f"Est. duration: {est_total:.1f} minutes")
    
    meta = os.path.join(DEST, "metadata_train.csv")
    if os.path.exists(meta):
        with open(meta) as f:
            print(f"Metadata entries: {len(f.readlines()) - 1}")
    else:
        print("WARNING: No metadata_train.csv found!")

In [ ]:
print("[4/7] Downloading XTTS v2 base model...")

from TTS.api import TTS

MODEL_NAME = "tts_models/multilingual/multi-dataset/xtts_v2"
tts = TTS(model_name=MODEL_NAME, progress_bar=True)
print("Base model ready.")

In [ ]:
print("[5/7] Configuring fine-tuning...")

import json

CONFIG_PATH = "/kaggle/working/xtts_finetune_config.json"

config = {
    "model": "xtts",
    "run_name": "xtts_v2_finetuned",
    "output_path": "/kaggle/working/xtts_finetuned",
    "datasets": [
        {
            "name": "voice_clone",
            "path": "/kaggle/working/voice_dataset",
            "meta_file_train": "metadata_train.csv",
            "language": "en",
        }
    ],
    "audio": {
        "sample_rate": 22050,
        "fft_size": 1024,
        "win_length": 1024,
        "hop_length": 256,
        "num_mels": 80,
    },
    "trainer": {
        "epochs": 50,
        "batch_size": 4,
        "eval_batch_size": 4,
        "mixed_precision": True,
        "save_step": 500,
        "print_step": 50,
        "plot_step": 500,
        "checkpoint": True,
        "lr": 5e-6,
        "lr_scheduler": "cosine",
        "warmup_steps": 100,
        "weight_decay": 1e-6,
        "grad_clip": 1.0,
    },
}

os.makedirs("/kaggle/working/xtts_finetuned", exist_ok=True)
with open(CONFIG_PATH, "w") as f:
    json.dump(config, f, indent=2)

print(f"Config saved. Output will go to: {config['output_path']}")
print(f"Epochs: {config['trainer']['epochs']}, Batch size: {config['trainer']['batch_size']}, LR: {config['trainer']['lr']}")

In [ ]:
print("[6/7] Starting fine-tuning...")

from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts
from TTS.tts.datasets import load_tts_samples
from TTS.tts.utils.text.tokenizer import VoiceBpeTokenizer
from TTS.trainer import Trainer, TrainingArgs
import json

with open(CONFIG_PATH) as f:
    cfg = json.load(f)

config = XttsConfig()
config.output_path = cfg["output_path"]
config.run_name = cfg["run_name"]
config.datasets = cfg["datasets"]
config.audio.sample_rate = cfg["audio"]["sample_rate"]

train_samples, eval_samples = load_tts_samples(
    config.datasets,
    eval_split=True,
    eval_split_max_size=4,
    eval_split_size=0.01,
)

print(f"Train samples: {len(train_samples)}")
print(f"Eval samples: {len(eval_samples)}")

model = Xtts.init_from_config(config)
model.init_multispeaker(config)

tokenizer = VoiceBpeTokenizer()

trainer = Trainer(
    args=TrainingArgs(
        output_path=cfg["output_path"],
        run_name=cfg["run_name"],
        epochs=cfg["trainer"]["epochs"],
        batch_size=cfg["trainer"]["batch_size"],
        eval_batch_size=cfg["trainer"]["eval_batch_size"],
        mixed_precision=cfg["trainer"]["mixed_precision"],
        save_step=cfg["trainer"]["save_step"],
        print_step=cfg["trainer"]["print_step"],
        plot_step=cfg["trainer"]["plot_step"],
        lr=cfg["trainer"]["lr"],
        lr_scheduler=cfg["trainer"]["lr_scheduler"],
        warmup_steps=cfg["trainer"]["warmup_steps"],
        weight_decay=cfg["trainer"]["weight_decay"],
        grad_clip=cfg["trainer"]["grad_clip"],
        checkpoint=cfg["trainer"]["checkpoint"],
    ),
    config=config,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
    training_assets={"tokenizer": tokenizer},
)

trainer.fit()
print("\nTraining complete!")

In [ ]:
print("[7/7] Testing and packaging model...")

import glob, shutil

from TTS.api import TTS

OUTPUT_DIR = "/kaggle/working/xtts_finetuned"
model_files = glob.glob(f"{OUTPUT_DIR}/**/*.pth", recursive=True)
config_files = glob.glob(f"{OUTPUT_DIR}/**/*.json", recursive=True)

if model_files:
    model_path = sorted(model_files)[-1]
    config_path = config_files[0] if config_files else None
    
    tts_ft = TTS(model_path=model_path, config_path=config_path, progress_bar=False)
    
    ref_wav = glob.glob("/kaggle/working/voice_dataset/wavs/*.wav")[0]
    tts_ft.tts_to_file(
        "Hello, this is my cloned voice from Kaggle training.",
        speaker_wav=ref_wav,
        language="en",
        file_path="/kaggle/working/test_output.wav"
    )
    print("Test inference successful.")
    
    from IPython.display import Audio
    Audio("/kaggle/working/test_output.wav")
else:
    print("No model files found.")

In [ ]:
# Package for download
PACKAGE = "/kaggle/working/xtts_finetuned_export"
os.makedirs(PACKAGE, exist_ok=True)

for f in glob.glob(f"{OUTPUT_DIR}/**/*.pth", recursive=True):
    shutil.copy2(f, os.path.join(PACKAGE, os.path.basename(f)))
for f in glob.glob(f"{OUTPUT_DIR}/**/*.json", recursive=True):
    shutil.copy2(f, os.path.join(PACKAGE, os.path.basename(f)))

zip_path = "/kaggle/working/xtts_finetuned_export"
shutil.make_archive(zip_path, "zip", PACKAGE)

size_mb = os.path.getsize(zip_path + ".zip") / 1024**2
print(f"Model packaged: {zip_path}.zip ({size_mb:.1f} MB)")
print("Download this file from the Kaggle Output tab!")